# Step 2 — Test-side noise injection

## Design
- Corrupt **test** utterances only (enrollment stays clean)
- SNRs: clean, 15 / 10 / 5 / 0 dB
- Prefer **MUSAN** (or any noise folder); if missing, use seeded **white noise**
- Fixed RNG seed for reproducibility

This notebook demos the mixer and optionally caches a few noisy WAVs.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import soundfile as sf
import torch

ROOT = Path.cwd().resolve()
if (ROOT / "noise_gated_lib.py").exists() is False:
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_LA,
    DEFAULT_SASV,
    DEFAULT_SNRS_DB,
    NOISY_DIR,
    ensure_dirs,
    load_waveform,
    maybe_noise_waveform,
    read_trials,
    resolve_audio_path,
    resolve_noise_bank,
    rms,
    snr_tag,
)

ensure_dirs()
SEED = 20260924
RNG = np.random.default_rng(SEED)

# Auto-discover MUSAN; or set explicitly, e.g. Path(r"D:/data/musan/noise")
NOISE_ROOT = Path(r"D:/downloads/musan/musan/noise")
noise_bank, used_root = resolve_noise_bank(NOISE_ROOT if str(NOISE_ROOT).strip() else None)
print("SNR grid:", list(map(snr_tag, DEFAULT_SNRS_DB)))


### Mix one enrollment-clean test clip at each SNR

In [ ]:
from experiment_lib import ensure_sasv_on_path

sasv = ensure_sasv_on_path(DEFAULT_SASV)
trials = read_trials(sasv, "dev", max_trials=5)
utt = trials[0].test_utt
split = "dev"
clean = load_waveform(resolve_audio_path(DEFAULT_LA, split, utt))
print("utt", utt, "samples", clean.numel(), "rms", rms(clean.numpy()))

demo_dir = NOISY_DIR / "demo" / utt
demo_dir.mkdir(parents=True, exist_ok=True)
sf.write(demo_dir / "clean.wav", clean.numpy(), 16000)

for snr in DEFAULT_SNRS_DB:
    noisy = maybe_noise_waveform(clean, snr_db=snr, noise_bank=noise_bank, rng=RNG)
    out = demo_dir / f"{snr_tag(snr)}.wav"
    sf.write(out, noisy.numpy(), 16000)
    print("wrote", out.name, "rms", float(rms(noisy.numpy())))

### Unit check: higher SNR ⇒ closer to clean (MSE)

In [ ]:
clean_np = clean.numpy()
rows = []
for snr in (15, 10, 5, 0):
    y = maybe_noise_waveform(clean, snr_db=snr, noise_bank=noise_bank, rng=RNG).numpy()
    mse = float(np.mean((y - clean_np) ** 2))
    rows.append({"snr_db": snr, "mse_vs_clean": mse})
rows

### Done when
- Demo WAVs exist under `cache/noisy_wavs/demo/`
- MSE increases as SNR decreases

Next → **03_baselines_and_gated_systems.ipynb**